### External Covariates: Financial Markets (Brent Crude Oil)

#### 1. What is "Brent" and what are its units?
**Brent Crude Oil** is the major trading classification of sweet light crude oil that serves as a major benchmark price for purchases of oil worldwide, particularly in Europe.
* **What it represents:** The price of a futures contract for oil delivery.
* **Unit of measurement:** US Dollars (USD) per barrel.
*(Note: Since oil is traded in USD and the Spanish economy operates in EUR, the real industrial impact depends on the combination of both. This justifies our later inclusion of the EUR/USD exchange rate).*

#### 2. What do Open, High, Low, and Close mean?
Financial markets are not open 24/7. Throughout the trading session, prices fluctuate. Financial datasets summarize this daily activity using four key data points (OHLC):
* **Open:** The price of the first trade when the market opens.
* **High:** The highest price the asset reached during the trading day.
* **Low:** The lowest price the asset reached during the trading day.
* **Close:** The final price at which the asset traded before the market closed.

#### 3. Why do we keep ONLY the "Close" column?
We drop the Open, High, and Low columns for two fundamental technical reasons regarding our Machine Learning model:
1.  **Market Consensus:** The closing price is the most critical value of the day. It represents the final agreement between buyers and sellers after processing all the news and events of that session. It is the standard reference value used by institutions.
2.  **Avoiding Multicollinearity (Noise):** The four columns are highly correlated. Passing all four to an algorithm (like XGBoost or a Neural Network) does not provide four times the information; it introduces mathematical redundancy. This can confuse the model and decrease its efficiency. Keeping only the "Close" price provides a single, clean, and representative variable.

In [38]:
import pandas as pd
import yfinance as yf

# --- 1. DOWNLOAD FINANCIAL DATA ---
print("Downloading Brent crude oil data...")
brent_df = yf.download('BZ=F', start='2014-12-30', end='2018-12-31')[['Close']]
brent_df.rename(columns={'Close': 'brent_close_usd'}, inplace=True)

print("Downloading EUR/USD data...")
eurusd_df = yf.download('EURUSD=X', start='2014-12-30', end='2018-12-31')[['Close']]
eurusd_df.rename(columns={'Close': 'eurusd_close'}, inplace=True)

# --- 2. WEEKEND HANDLING (Forward Fill) ---
# We create an index with ALL calendar days to fill the gaps.
all_days = pd.date_range(start='2014-12-30', end='2018-12-31', freq='D')

# Reindex both dataframes
brent_df = brent_df.reindex(all_days)
eurusd_df = eurusd_df.reindex(all_days)

# Fill NaN values on Saturdays and Sundays using Friday's closing price
brent_df['brent_close_usd'] = brent_df['brent_close_usd'].ffill()
eurusd_df['eurusd_close'] = eurusd_df['eurusd_close'].ffill()

# --- 3. DATA LEAKAGE PREVENTION (1-Day Shift) ---
# CRITICAL: We shift all prices one day forward to avoid data leakage.
brent_df['brent_close_usd_lag1'] = brent_df['brent_close_usd'].shift(1)
eurusd_df['eurusd_close_lag1'] = eurusd_df['eurusd_close'].shift(1)

# Drop the original un-lagged columns to guarantee they aren't used by mistake
brent_df = brent_df.drop(columns=['brent_close_usd'])
eurusd_df = eurusd_df.drop(columns=['eurusd_close'])

# --- 4. COMBINE AND FEATURE ENGINEERING ---
# Join both dataframes based on their date index
macro_df = brent_df.join(eurusd_df)

# Feature Engineering: Calculate Brent price in Euros (since Spain uses EUR)
# Brent (USD) / (USD per EUR) = Brent (EUR)
macro_df['brent_close_eur_lag1'] = macro_df['brent_close_usd_lag1'] / macro_df['eurusd_close_lag1']

# Drop the initial NaN row created by the shift() and any other missing values
macro_df = macro_df.dropna()

print("\nFinal Macroeconomic DataFrame:")
display(macro_df.head())

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


Final Macroeconomic DataFrame:


Price,brent_close_usd_lag1,eurusd_close_lag1,brent_close_eur_lag1
Ticker,,,
2014-12-31,57.900002,1.215347,47.640700
2015-01-01,57.330002,1.216205,47.138445
2015-01-02,57.330002,1.209863,47.385539
2015-01-03,56.419998,1.208941,46.668929
2015-01-04,56.419998,1.208941,46.668929
